# Original persona recipe
Enable Internet and a T4. Install the separate pinned persona environment and restart the kernel before the training cell if any incompatible libraries were already imported. This is a reproduction recipe, not a new result. The selected adapter is best-by-validation-accuracy; final is retained separately.

In [ ]:
from pathlib import Path
import subprocess,sys,os,json
ROOT=Path('/kaggle/working/cs5326-pa2-small-model-alignment')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/salehfarooq/cs5326-pa2-small-model-alignment.git',str(ROOT)],check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH']=str(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-persona.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.','--no-deps'],check=True)


In [ ]:
import torch
from importlib.metadata import version
assert torch.cuda.device_count()>=1
print({name:version(name) for name in ['torch','transformers','datasets','trl','peft']})
REVISION='a10cc1512eabd3dde888204e902eca88bddb4951'
RUN=Path('/kaggle/working/persona_reproduction')
subprocess.run([sys.executable,'-u','-m','scripts.train_persona_lora',
  '--data_path',str(ROOT/'data/persona_lora/ramsay_style_preferences.json'),
  '--output_dir',str(RUN),'--revision',REVISION],check=True,env={**os.environ,'CUDA_VISIBLE_DEVICES':'0'})
for label,adapter in [('baseline',None),('best',RUN/'best_adapter'),('final',RUN/'final_adapter')]:
    cmd=[sys.executable,'-u','-m','scripts.generate_persona','--bank',str(ROOT/'evaluation/persona_eval_40.json'),
         '--output',str(Path('/kaggle/working')/('persona-'+label+'-reproduction')),'--revision',REVISION]
    if adapter:cmd+=['--adapter',str(adapter)]
    subprocess.run(cmd,check=True)
